# WSI DeID Batch Export

Use this notebook when you have a folder of whole-slide images, for example 3 `.ndpi` files on the Desktop, and want to create clean lossless de-identified derivatives.

The original WSI files are not modified.

## 1. Load WSI DeID

Run this cell first. If `wsi-deid` is not installed in the current Jupyter kernel, it will install the local repository automatically.

In [ ]:
import sys
import subprocess
from pathlib import Path
from datetime import datetime


def find_repo_dir(start: Path) -> Path:
    """Find the repository folder containing pyproject.toml."""
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "wsi_deid").exists():
            return candidate
    raise RuntimeError(
        "Could not find the wsi-deid repository folder. "
        "Open this notebook from inside the extracted/cloned wsi-deid folder."
    )


REPO_DIR = find_repo_dir(Path.cwd())
print("Repository folder:", REPO_DIR)

try:
    from wsi_deid.export import export_clean_wsi
    print("wsi-deid is ready.")
except ModuleNotFoundError:
    print("Installing wsi-deid into this Jupyter environment...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        str(REPO_DIR),
    ])
    from wsi_deid.export import export_clean_wsi
    print("wsi-deid is installed and ready.")

SUPPORTED_EXTENSIONS = {
    ".ndpi", ".svs", ".scn", ".mrxs", ".tif", ".tiff",
    ".ome.tif", ".ome.tiff", ".vms", ".vmu", ".bif", ".dcm",
}


In [ ]:
# This cell is intentionally left empty. The setup/import code is now in the cell above.


## 2. Set input and output folders

Change `INPUT_DIR` to the folder on your Desktop containing the 3 WSI files.

In [ ]:
# Example: if your slides are in C:\Users\HP\Desktop\test_wsi
INPUT_DIR = Path(r'C:\Users\HP\Desktop\test_wsi')

# A new output folder is created for each run, so old results are not overwritten.
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path(r'C:\Users\HP\Desktop') / f'wsi_deid_clean_{RUN_ID}'
QC_DIR = OUTPUT_DIR / 'qc'
KEY_CSV = OUTPUT_DIR / 'pseudonym_key.csv'

print('Input folder:', INPUT_DIR)
print('Output folder:', OUTPUT_DIR)
print('Key CSV:', KEY_CSV)

## 3. Find WSI files

In [ ]:
def is_supported_wsi(path: Path) -> bool:
    name = path.name.lower()
    return any(name.endswith(ext) for ext in SUPPORTED_EXTENSIONS)


slides = sorted(p for p in INPUT_DIR.rglob('*') if p.is_file() and is_supported_wsi(p))

print('Found WSI files:', len(slides))
for i, slide in enumerate(slides, start=1):
    print(f'{i:02d}. {slide}')

## 4. Batch export clean lossless WSI files

This creates one clean `.ome.tif` per input slide. Defaults:

- lossless `deflate` compression
- barcode/label end masked: 22 mm
- handwritten/scratch end masked: 12 mm
- original associated macro/label images omitted from the clean WSI
- QC before/mask/after PNGs saved separately

In [ ]:
BARCODE_END_MM = 22.0
HANDWRITTEN_END_MM = 12.0
MAX_SUBLEVELS = 6
COMPRESSION = 'deflate'  # lossless. Do not use 'jpeg' if avoiding pathology information loss.

results = []

if not slides:
    raise RuntimeError('No WSI files found. Check INPUT_DIR and file extensions.')

for index, slide_path in enumerate(slides, start=1):
    study_id = f'WSI_{index:06d}'
    print('\n' + '=' * 80)
    print(f'Processing {slide_path.name} -> {study_id}')

    result = export_clean_wsi(
        source=slide_path,
        output_dir=OUTPUT_DIR,
        qc_dir=QC_DIR,
        key_csv=KEY_CSV,
        study_id=study_id,
        barcode_end_mm=BARCODE_END_MM,
        handwritten_end_mm=HANDWRITTEN_END_MM,
        max_sublevels=MAX_SUBLEVELS,
        compression=COMPRESSION,
    )
    results.append(result)
    print('Clean WSI:', result['output_path'])

print('\nDone')
print('Output folder:', OUTPUT_DIR)
print('QC folder:', QC_DIR)
print('Key CSV:', KEY_CSV)

## 5. Review outputs

In [ ]:
print('Clean WSI files:')
for p in sorted(OUTPUT_DIR.glob('*.ome.tif')):
    print(p.name, round(p.stat().st_size / (1024 ** 3), 2), 'GB')

print('\nQC files:')
for p in sorted(QC_DIR.glob('*')):
    print(p.name)

print('\nKey CSV:')
print(KEY_CSV)

## 6. Visual QC checklist

For each slide, open the QC images and confirm:

- barcode/label end is fully blocked
- handwritten/scratch-number end is fully blocked
- diagnostic tissue is not blocked
- output `.ome.tif` opens in QuPath/Fiji or your chosen viewer
- `pseudonym_key.csv` is stored securely and not shared with the clean WSI folder